MIT License

Copyright (c) 2026 THALES

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Cost-Model Path Divergence — Fig. 5

Paper-facing skeleton that produces Fig. 5 (same tile, same start and goal, five cost models overlaid) for the NeurIPS 2026 submission. The developer walkthrough at `../../notebooks/costmodel.ipynb` remains the long-form exploration surface; this notebook is the paper-facing export.

Source of truth for paper structure: `../../../passage-paper/neurips/skeleton.md` (v3).

## Paper Mapping

| Claim | `\evidence{}` tag | Output artifact |
| --- | --- | --- |
| Sec. 5.2 the five physical cost models induce qualitatively different optimal paths | `passage/paper/notebooks/cost_model_path_divergence.ipynb -> passage/paper/figures/cost_model_path_divergence.pdf` | `figures/cost_model_path_divergence.pdf` |
| Sec. 5.2 pairwise normalised Hausdorff divergence between cost-model paths | `passage/paper/notebooks/cost_model_path_divergence.ipynb -> cost_model_path_divergence.csv` | `figures/cost_model_path_divergence.csv` |

## Inputs / Outputs

| Kind | Path |
| --- | --- |
| Input | One reference elevation tile and one fixed `(start, goal)` pair |
| Input | Cost models `{distance, elevation, absolute_elevation, slope_uphill, energy}` with per-model log-uniform weight draws |
| Output | `../figures/cost_model_path_divergence.pdf` (+ `.png`) |
| Output | `../figures/cost_model_path_divergence.csv` |
| Output | `../figures/cost_model_path_divergence.manifest.json` |


In [ ]:
# Setup: import helpers and handle optional dependencies.
from __future__ import annotations

import sys
from pathlib import Path

PAPER_DIR = Path.cwd().resolve().parent
if str(PAPER_DIR) not in sys.path:
    sys.path.insert(0, str(PAPER_DIR))

try:
    from helpers import FIGURES_DIR, emit_manifest, run_astar_bench, save_figure
except ImportError as exc:  # pragma: no cover
    raise ImportError(
        "Could not import passage.paper.helpers; run from passage/paper/notebooks/."
    ) from exc

try:
    import numpy as np
    import matplotlib.pyplot as plt
    HAS_PLOTTING = True
except ImportError:
    HAS_PLOTTING = False

try:
    import passage  # noqa: F401
    HAS_PASSAGE = True
except ImportError:
    HAS_PASSAGE = False

STUDY_ID = "cost_model_path_divergence"
RESOLUTION = 256
COST_MODELS = [
    "distance",
    "elevation",
    "absolute_elevation",
    "slope_uphill",
    "energy",
]
WEIGHTS = {
    "distance": 0.0,
    "elevation": 1.0,
    "absolute_elevation": 1.0,
    "slope_uphill": 5.0,
    "energy": 5.0,
}
BACKEND = "passage_grid"
HEURISTIC = "euclidean"
TILE_SEED = 42
START = (16, 16)
GOAL = (RESOLUTION - 16, RESOLUTION - 16)

In [ ]:
# Run one A* call per cost model on the same tile and (start, goal) pair.
#
# TODO: load a real tile via helpers.load_tile instead of synthesising a
# stub grid here. The developer walkthrough at
# ../../notebooks/costmodel.ipynb carries the full cost-model parameter
# exploration.

rows = []
if HAS_PLOTTING:
    rng = np.random.default_rng(TILE_SEED)
    grid = rng.random((RESOLUTION, RESOLUTION)).astype("float32")
    for cost_model in COST_MODELS:
        result = run_astar_bench(
            backend=BACKEND,
            grid=grid,
            start=START,
            goal=GOAL,
            cost_model=cost_model,
            weight=WEIGHTS[cost_model],
            heuristic=HEURISTIC,
        )
        rows.append(result.to_dict())

csv_path = FIGURES_DIR / f"{STUDY_ID}.csv"
if rows:
    header = list(rows[0].keys())
    lines = [",".join(header)]
    for row in rows:
        lines.append(",".join(str(row[k]) for k in header))
    csv_path.write_text("\n".join(lines) + "\n")
else:
    csv_path.write_text("cost_model,resolution,wall_clock_s,status\n")
print(f"wrote {csv_path}")

In [ ]:
# Build the overlay figure (Fig. 5).
#
# The final paper figure should overlay the five A* paths on the same
# elevation tile, colour-coded by cost model, and include a small pairwise
# normalised Hausdorff matrix next to the panel. Here we emit a stub
# placeholder panel so the artifact path is populated even without real
# paths returned from the backend.
if HAS_PLOTTING:
    fig, (ax_overlay, ax_matrix) = plt.subplots(1, 2, figsize=(9.0, 4.0))

    ax_overlay.imshow(grid, cmap="terrain")
    ax_overlay.scatter(*START[::-1], c="lime", s=30, label="start")
    ax_overlay.scatter(*GOAL[::-1], c="red", s=30, label="goal")
    ax_overlay.set_title("A* paths per cost model (stub — overlay real paths)")
    ax_overlay.set_xticks([])
    ax_overlay.set_yticks([])
    ax_overlay.legend(loc="lower right", fontsize=8)

    n = len(COST_MODELS)
    matrix = np.zeros((n, n))
    im = ax_matrix.imshow(matrix, cmap="magma")
    ax_matrix.set_xticks(range(n))
    ax_matrix.set_yticks(range(n))
    ax_matrix.set_xticklabels(COST_MODELS, rotation=45, ha="right", fontsize=8)
    ax_matrix.set_yticklabels(COST_MODELS, fontsize=8)
    ax_matrix.set_title("pairwise normalised Hausdorff (stub)")
    fig.colorbar(im, ax=ax_matrix, fraction=0.046, pad=0.04)
    fig.tight_layout()
else:
    fig = None

In [ ]:
# Emit manifest and save figure.
outputs = [csv_path]
if fig is not None:
    pdf_path = save_figure(fig, study_id=STUDY_ID)
    outputs.extend([pdf_path, pdf_path.with_suffix(".png")])

manifest_path = emit_manifest(
    study_id=STUDY_ID,
    inputs={
        "resolution": RESOLUTION,
        "cost_models": COST_MODELS,
        "weights": WEIGHTS,
        "backend": BACKEND,
        "heuristic": HEURISTIC,
        "tile_seed": TILE_SEED,
        "start": list(START),
        "goal": list(GOAL),
        "has_plotting": HAS_PLOTTING,
        "has_passage_package": HAS_PASSAGE,
        "paper_section": "5.2",
        "paper_anchors": ["fig:cost-models"],
    },
    outputs=outputs,
)
print(f"wrote {manifest_path}")